# Silver events and versioned tariffs

Use explicit schemas and half-open SCD Type 2 intervals. Repeated prices collapse into a single version.


In [ ]:
if "lake_root" not in globals():
    dbutils.widgets.text("lake_root", "/Volumes/workspace/default/finops/lake")
    lake_root = dbutils.widgets.get("lake_root").rstrip("/")

from pyspark.sql import Window
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")
event_schema = (
    "event_id string, occurred_at timestamp, tenant_id string, workflow string, model_id string, "
    "data_kind string, source string, status string, input_tokens long, output_tokens long, "
    "latency_ms double, ttft_ms double, charged_usd string, experiment_id string, arm string, "
    "ground_truth string, prediction string, fallback boolean"
)
price_schema = "model_id string, observed_at timestamp, input_usd string, output_usd string, source string"
score_schema = "source string, model_id string, benchmark string, score double, observed_at timestamp, revision string"

def parsed(kind, schema, keys):
    rows = spark.read.format("delta").load(f"{lake_root}/bronze/{kind}").select(
        F.from_json("payload", schema, {"mode": "FAILFAST"}).alias("record")
    ).select("record.*").dropDuplicates()
    any_missing = F.col(keys[0]).isNull()
    for key in keys[1:]:
        any_missing = any_missing | F.col(key).isNull()
    if rows.filter(any_missing).limit(1).count():
        raise ValueError(f"Missing key in {kind}")
    if rows.groupBy(*keys).count().filter(F.col("count") > 1).limit(1).count():
        raise ValueError(f"Conflicting keys in {kind}")
    return rows

events = parsed("events", event_schema, ["event_id"]).withColumn(
    "event_date", F.to_date("occurred_at")
).withColumn("charged_usd", F.col("charged_usd").cast("decimal(36,18)"))
prices = parsed("prices", price_schema, ["source", "model_id", "observed_at"]).withColumnRenamed(
    "source", "price_source"
).withColumns({column: F.col(column).cast("decimal(24,18)") for column in ("input_usd", "output_usd")})
scores = parsed("scores", score_schema, ["source", "model_id", "benchmark", "observed_at", "revision"])
ordered = Window.partitionBy("price_source", "model_id").orderBy("observed_at")
changes = prices.withColumns({
    "previous_input": F.lag("input_usd").over(ordered),
    "previous_output": F.lag("output_usd").over(ordered),
}).filter(
    ~(F.col("input_usd").eqNullSafe(F.col("previous_input"))
      & F.col("output_usd").eqNullSafe(F.col("previous_output")))
).drop("previous_input", "previous_output")
versions = changes.withColumn("valid_to", F.lead("observed_at").over(ordered)).withColumnRenamed(
    "observed_at", "valid_from"
).withColumn("price_version_id", F.sha2(F.to_json(F.struct("price_source", "model_id", "valid_from")), 256))
for name, frame in [("events", events), ("price_versions", versions), ("scores", scores)]:
    frame.write.format("delta").mode("overwrite").save(f"{lake_root}/silver/{name}")
    print(name, spark.read.format("delta").load(f"{lake_root}/silver/{name}").count())
